# 03 · Supervise an agent campaign (control scientist)

You start a **scripted** agent session, watch it, and act as its supervisor at the CPU gate.
Exercise E48 in the platform manual follows this notebook.

* `medcea_agent` talks to the research orchestrator. You sign in once with a device code; the
  token lives only in this kernel's memory.
* Every turn, tool call, gate decision and the stop is an event on the session's hash-chained
  trace (database `medcea_research`, schema `agent_trace`), with **your** sub and role on it.
* This folder is read-only: copy the notebook to your home folder to change it.

In [ ]:
from medcea_agent import Client, CliError

c = Client()
c.signin()          # open the link, sign in, confirm the code; the cell waits

## 1 · A campaign that runs without asking

`compare_controllers` lists the lake's snapshots, estimates a small campaign (under 2 CPU-hours,
so the gate says `auto`), runs it as a simulation Job, and proposes a PID setting. `c.watch`
prints each event until the session stops.

In [ ]:
sid = c.start("compare_controllers")["sid"]
print(sid)
c.watch(sid)

In [ ]:
c.replay(sid)       # strict replay as a Job: expect {'ok': True, ...}

## 2 · A campaign that waits for you

`gate_check` asks for six runs of a 9,000-hour fruiting scenario: about 3 CPU-hours. Above
2 CPU-hours the simulation server does not start it; it records `gate_pending` and the agent
waits (wall time, not turns) until you approve or deny.

The next cell starts the session and waits until the gate appears, then prints the gate id.
Do **not** use `c.watch` here: it would hold this kernel and you could not answer the gate.

In [ ]:
import time

sid2 = c.start("gate_check")["sid"]
print("session", sid2)

gid, seen = None, 0
for _ in range(120):                       # up to 10 minutes
    for e in c.events(sid2, seen):
        seen = e["event_id"]
        print(Client.line(e))
        if e["event"] == "gate_pending":
            gid = e["payload"]["gid"]
    if gid:
        break
    time.sleep(5)
print("gate", gid)

Give the bio scientist `sid2` and `gid` now if you are running E48: their approval must be
refused (notebook 04). Then approve it yourself, **with a small cap**: the simulation Job is
stopped by its watchdog once it has used 0.05 CPU-hours, and the agent gets that as a failed
tool result. (Approving without `cap_h` runs the whole campaign, about 3 CPU-hours;
`"deny"` sends the agent "the supervisor denied this campaign".)

In [ ]:
c.answer(sid2, "gates", gid, "approve", note="E48: approved with a small cap", cap_h=0.05)

In [ ]:
c.watch(sid2)       # until the stop: expect the Job ended at its cap, then reason finished

In [ ]:
c.info(sid2)        # budget and what was spent

## Afterwards

`c.stop(sid)` ends a session that is still active. The trace stays: it is never deleted.